# Domux safety-gate experiment

This notebook runs a real pinned Hugging Face snapshot of `iFlytekOpenSource/Domux` on 48 public synthetic smart-home safety cases. It stores raw model outputs and aggregate metrics, but never stores or prints the Hugging Face token. Use a GPU runtime.

In [ ]:
!nvidia-smi
!python --version
!pip -q install 'transformers>=5.0.0' 'accelerate>=1.10.0' 'bitsandbytes>=0.49.0' 'huggingface_hub>=1.0.0'

## Hugging Face authentication

Accept the Gemma terms on the Domux model page first. The widget below asks you to paste a read-only Hugging Face token directly into the notebook session. Do not paste the token into any code cell, output, screenshot, Discussion, or GitHub file.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!git clone --depth 1 --branch case/domux-safety-gate https://github.com/yangmengze608-afk/domux.git /content/domux
%cd /content/domux/cases/domux-safety-gate
!python -m unittest -v test_safety_gate.py test_dataset.py test_evaluate_safety.py

## Smoke test

Run two samples first. If the free GPU cannot load the NF4 model, stop here; do not switch to a paid runtime without explicit approval.

In [ ]:
!python run_transformers.py --dataset example_safety_commands.jsonl --output results/smoke.jsonl --quantization nf4 --limit 2 --warmup 1
!sed -n '1,2p' results/smoke.jsonl
!cat results/smoke.metadata.json

## Full 48-case run and evaluation

In [ ]:
!python run_transformers.py --dataset example_safety_commands.jsonl --output results/domux_raw.jsonl --quantization nf4 --warmup 2
!python evaluate_safety.py --dataset example_safety_commands.jsonl --responses results/domux_raw.jsonl --output results/safety_report.json

In [ ]:
import json
from pathlib import Path
report = json.loads(Path('results/safety_report.json').read_text())
{key: value for key, value in report.items() if key != 'details'}

In [ ]:
import shutil
shutil.make_archive('/content/domux-safety-gate-results', 'zip', 'results')
print('/content/domux-safety-gate-results.zip contains logs and metrics only; no model weights or token.')